In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sktime.split import ExpandingWindowSplitter
import joblib
import json
import optuna
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from features.FeatureExtractor import FeatureExtractor
from features.TargetEncoder import TargetEncoder
from statsmodels.tsa.stattools import adfuller

In [2]:
cleaned_dataset_address = "dataset/interim/past_dataset.csv"
SEED = 99
TARGET = "general_dam_occupancy_rate"

In [3]:
try:
    with open("models/best_features_by_lasso.json", errors="FileNotFoundError") as f:
        best_features = json.load(f)
        
except FileNotFoundError:
    best_features = []

In [4]:
best_features

['general_dam_occupancy_rate_lag_1',
 'general_dam_occupancy_rate_rw2_mean',
 'general_dam_occupancy_rate_rw2_max']

In [5]:
past_knowledge = (
    pd.read_csv(
        cleaned_dataset_address,
        parse_dates=["datetime"],
        converters={"weather_code": str},
    )
    .set_index("datetime")
    .sort_index()
)


In [6]:
parameters = {
    "past_knowledge": past_knowledge,
    "cyclical_feature_names": {
        "month": 12,
        "day": 31,
        "day_of_year": 365,
        "week_of_year": 52,
        "quarter": 4,
        # "season": 4,
        "is_weekend": 2,
        "precipitation_hours": 24,
    },
    "lag_size": 30,
    "window_size": 30,
}

In [7]:
feature_extractor = FeatureExtractor(**parameters)

In [8]:
known_dates = past_knowledge.index

In [9]:
y_values = past_knowledge.loc[:, TARGET]

In [10]:
p_value = adfuller(y_values)[1]

if p_value >= 0.05:
    print("We can't deny that dataset is non-stationary")
elif p_value < 0.05:
    print("We can deny that dataset is non-stationary")


We can deny that dataset is non-stationary


In [11]:
if best_features:
    X_values = feature_extractor.transform(known_dates).filter(items=[*best_features])
else:
    X_values = feature_extractor.transform(known_dates)

In [12]:
train_size = int(len(X_values) * 0.8)

In [13]:
train_df = X_values.iloc[:train_size].merge(y_values, on="datetime", how="inner")
test_df = X_values.iloc[train_size:].merge(y_values, on="datetime", how="inner")

In [14]:
X_train, y_train = (
    train_df.drop(columns=["general_dam_occupancy_rate"]),
    train_df["general_dam_occupancy_rate"],
)

X_test, y_test = (
    test_df.drop(columns=["general_dam_occupancy_rate"]),
    test_df["general_dam_occupancy_rate"],
)


In [15]:
def get_expending_window_splitter(df: pd.DataFrame, n_fold: int = 5):
    initial_window = len(df) // 2
    step_length = (len(df) - initial_window) // n_fold
    fh = np.arange(1, step_length + 1)

    return ExpandingWindowSplitter(
        initial_window=initial_window, step_length=step_length, fh=fh
    )


In [16]:
def winsorize_iqr(
    df: pd.DataFrame,
    lower_quantile: float = 0.25,
    upper_quantile: float = 0.75,
    multiplier: float = 1.5,
) -> pd.DataFrame:
    if not (0 <= lower_quantile < upper_quantile <= 1):
        raise ValueError("Quantiles must satisfy 0 <= lower < upper <= 1")
    if multiplier <= 0:
        raise ValueError("Multiplier must be positive")
    
    df = df.copy()
    numerical_cols = df.select_dtypes(include=["number"]).columns

    for col in numerical_cols:
        Q1 = df[col].quantile(lower_quantile)
        Q3 = df[col].quantile(upper_quantile)
        IQR = Q3 - Q1

        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR

        is_col_integer = pd.api.types.is_integer_dtype(df[col])
        is_col_float = pd.api.types.is_float_dtype(df[col])
        
        if is_col_integer:
            df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
        elif is_col_float:
            df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

    return df


In [17]:
X_train = winsorize_iqr(X_train)
X_test = winsorize_iqr(X_test)

## Feature Selection

In [18]:
if len(best_features) == 0:
    splitter = get_expending_window_splitter(X_train)

    te = TargetEncoder(cv=splitter)
    X_train_fs = te.fit_transform(X_train, pd.DataFrame(y_train))
    X_test_fs = te.transform(X_test)

    ss = StandardScaler().set_output(transform="pandas")
    X_train_fs_scaled = ss.fit_transform(X_train_fs)
    X_test_fs_scaled = ss.transform(X_test_fs)

    lasso = Lasso(random_state=SEED)
    model = lasso.fit(X_train_fs_scaled, y_train)

    feature_importance = (
        pd.DataFrame(lasso.coef_, index=lasso.feature_names_in_, columns=["coef"])
        .abs()
        .sort_values("coef", ascending=False)
    )

    best_cols = feature_importance.loc[
        feature_importance["coef"] > 0
    ].index.to_list()
    
    with open("models/best_features_by_lasso.json", "w") as f:
        json.dump(best_cols, f)


In [19]:
def optimize_ridge(trial: optuna.Trial):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-5, 1e3),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "solver": trial.suggest_categorical(
            "solver", ["auto", "cholesky", "lsqr", "sag"]
        ),
    }

    splitter = get_expending_window_splitter(X_train)
    results = []

    for train_ind, val_ind in splitter.split(y_train):
        X_train_fold, X_val_fold = X_train.iloc[train_ind], X_train.iloc[val_ind]
        y_train_fold, y_val_fold = y_train.iloc[train_ind], y_train.iloc[val_ind]

        te_cv = get_expending_window_splitter(X_train_fold)
        te = TargetEncoder(cv=te_cv)
        X_train_fold = te.fit_transform(X_train_fold, pd.DataFrame(y_train_fold))
        X_val_fold = te.transform(X_val_fold)

        ss = StandardScaler().set_output(transform="pandas")
        X_train_scaled = ss.fit_transform(X_train_fold)
        X_val_scaled = ss.transform(X_val_fold)

        model = Ridge(**params, random_state=SEED)
        model.fit(X_train_scaled, y_train_fold)
        y_pred = model.predict(X_val_scaled)
        results.append(mean_absolute_error(y_val_fold, y_pred))

    dummy_forecaster_mae_error = 14.062895669528846

    return np.mean(results) if np.mean(results) < dummy_forecaster_mae_error else np.inf


In [20]:
study = optuna.create_study(direction="minimize", storage="sqlite:///example-study.db")
study.optimize(optimize_ridge, n_trials=2000, show_progress_bar=True, n_jobs=-1)

[I 2025-03-06 14:40:16,304] A new study created in RDB with name: no-name-47cc4b4f-3d38-4212-9660-b43ddc8d89f4


  0%|          | 0/2000 [00:00<?, ?it/s]

[I 2025-03-06 14:40:17,883] Trial 4 finished with value: inf and parameters: {'alpha': 774.7555219635001, 'fit_intercept': False, 'solver': 'cholesky'}. Best is trial 4 with value: inf.
[I 2025-03-06 14:40:17,984] Trial 2 finished with value: 0.8183825556383635 and parameters: {'alpha': 438.1037099244699, 'fit_intercept': True, 'solver': 'cholesky'}. Best is trial 2 with value: 0.8183825556383635.
[I 2025-03-06 14:40:18,291] Trial 1 finished with value: 1.5444382234272205 and parameters: {'alpha': 885.1915372328981, 'fit_intercept': True, 'solver': 'lsqr'}. Best is trial 2 with value: 0.8183825556383635.
[I 2025-03-06 14:40:18,294] Trial 3 finished with value: inf and parameters: {'alpha': 198.0328933649226, 'fit_intercept': False, 'solver': 'sag'}. Best is trial 2 with value: 0.8183825556383635.
[I 2025-03-06 14:40:18,298] Trial 7 finished with value: 1.2102923481830796 and parameters: {'alpha': 675.6226516705339, 'fit_intercept': True, 'solver': 'auto'}. Best is trial 2 with value: 0

/opt/anaconda3/envs/ds-study/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/ds-study/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[I 2025-03-06 14:41:20,872] Trial 920 finished with value: 0.19893605944181744 and parameters: {'alpha': 38.83834515623717, 'fit_intercept': True, 'solver': 'auto'}. Best is trial 444 with value: 0.028335061292146406.
[I 2025-03-06 14:41:20,896] Trial 921 finished with value: 0.19861793482730267 and parameters: {'alpha': 38.148279380791216, 'fit_intercept': True, 'solver': 'sag'}. Best is trial 444 with value: 0.028335061292146406.
[I 2025-03-06 14:41:20,984] Trial 923 finished with value: 0.19710829878957647 and parameters: {'alpha': 36.51874183891525, 'fit_intercept': True, 'solver': 'sag'}. Best is trial 444 with value: 0.028335061292146406.
[I 2025-03-06 14:41:20,994] Trial 922 finished with value: 0.19795759849146793 and parameters: {'alpha': 37.81929716734245, 'fit_intercept': True, 'solver': 'auto'}. Best is trial 444 with value: 0.028335061292146406.
[I 2025-03-06 14:41:21,028] Trial 925 finished with value: 0.1963774415989926 and parameters: {'alpha': 36.15580054901287, 'fit_i

...

[I 2025-03-06 14:42:35,637] Trial 1963 finished with value: 0.180822771381579 and parameters: {'alpha': 18.11299669746753, 'fit_intercept': True, 'solver': 'auto'}. Best is trial 1109 with value: 0.018637291002775534.
[I 2025-03-06 14:42:35,732] Trial 1967 finished with value: 0.18155426218133228 and parameters: {'alpha': 17.69230094615387, 'fit_intercept': True, 'solver': 'sag'}. Best is trial 1109 with value: 0.018637291002775534.
[I 2025-03-06 14:42:35,778] Trial 1965 finished with value: 0.12190003842002475 and parameters: {'alpha': 0.5396615053674965, 'fit_intercept': True, 'solver': 'auto'}. Best is trial 1109 with value: 0.018637291002775534.
[I 2025-03-06 14:42:35,795] Trial 1964 finished with value: 0.18117896638569086 and parameters: {'alpha': 17.14486726737753, 'fit_intercept': True, 'solver': 'sag'}. Best is trial 1109 with value: 0.018637291002775534.


/opt/anaconda3/envs/ds-study/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/ds-study/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[I 2025-03-06 14:42:35,894] Trial 1966 finished with value: 0.18133015284488085 and parameters: {'alpha': 18.750009305619642, 'fit_intercept': True, 'solver': 'auto'}. Best is trial 1109 with value: 0.018637291002775534.
[I 2025-03-06 14:42:35,967] Trial 1969 finished with value: 0.18193386487458887 and parameters: {'alpha': 17.929136127787487, 'fit_intercept': True, 'solver': 'sag'}. Best is trial 1109 with value: 0.018637291002775534.


/opt/anaconda3/envs/ds-study/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/ds-study/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[I 2025-03-06 14:42:36,116] Trial 1970 finished with value: 0.1802022299676193 and parameters: {'alpha': 15.691015283646722, 'fit_intercept': True, 'solver': 'sag'}. Best is trial 1109 with value: 0.018637291002775534.
[I 2025-03-06 14:42:36,126] Trial 1971 finished with value: 0.1793011635228208 and parameters: {'alpha': 16.186911282600946, 'fit_intercept': True, 'solver': 'auto'}. Best is trial 1109 with value: 0.018637291002775534.
[I 2025-03-06 14:42:36,235] Trial 1972 finished with value: 0.22661726035670235 and parameters: {'alpha': 65.02510041024576, 'fit_intercept': True, 'solver': 'auto'}. Best is trial 1109 with value: 0.018637291002775534.
[I 2025-03-06 14:42:36,253] Trial 1973 finished with value: 0.2288940365439968 and parameters: {'alpha': 67.02851328067216, 'fit_intercept': True, 'solver': 'auto'}. Best is trial 1109 with value: 0.018637291002775534.
[I 2025-03-06 14:42:36,281] Trial 1968 finished with value: 0.03307594180458474 and parameters: {'alpha': 0.00073848084232

In [21]:
ridge = Ridge(**study.best_params, random_state=SEED)

In [22]:
cv = get_expending_window_splitter(y_train)
te = TargetEncoder(cv=cv)
X_train = te.fit_transform(X_train, pd.DataFrame(y_train))
X_test = te.transform(X_test)

In [23]:
ss = StandardScaler().set_output(transform="pandas")
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)


In [24]:
model = ridge.fit(X_train_scaled, y_train)

In [25]:
joblib.dump(model, "models/ridge-regression-1.pkl.gz", compress="gzip")

['models/ridge-regression-1.pkl.gz']

In [26]:
train_pred = model.predict(X_train_scaled)
mean_absolute_error(y_train, train_pred)

0.013298685816714558

In [27]:
test_pred = model.predict(X_test_scaled)

In [28]:
mean_absolute_error(y_test, test_pred)

0.015979958736535255

In [29]:
train_pred_series = pd.Series(train_pred, name="predictions", index=y_train.index)
test_pred_series = pd.Series(test_pred, name="predictions", index=y_test.index)

train_pred_df = pd.concat([train_pred_series, y_train], axis=1)
test_pred_df = pd.concat([test_pred_series, y_test], axis=1)

In [30]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Train", "Test"))

for i, (df, name) in enumerate(zip([train_pred_df, test_pred_df], ["Train", "Test"])):
    fig.add_trace(go.Scatter(x=df.index, y=df[TARGET], mode="lines", name=f"{name} True"), row=1, col=i+1)
    fig.add_trace(go.Scatter(x=df.index, y=df["predictions"], mode="lines", name=f"{name} Predictions"), row=1, col=i+1)

fig.update_layout(height=400, width=1200, title_text="Predictions vs. True Values")
fig.show()
